# 04 · Biological identity: do labels agree with marker genes? (Finding 2)
For every labelled population, score CellMarker 2.0 markers of each candidate cell type. A label is *supported* if its own markers score highest.

In [ ]:
import pandas as pd
import scanpy as sc

cm = pd.read_excel("../data/raw/cellmarker2_human.xlsx")
cm.columns.tolist()   # look for the tissue, cell name, Cell Ontology ID and marker columns

In [ ]:
# Filter to blood, map to CL IDs, keep markers reported by >= 2 sources.
# Fill in column names after inspecting the table above.
# markers = {cl_id: [gene, ...], ...}
markers = {}

In [ ]:
def marker_support(adata, markers, label_col="cell_type_ontology_term_id"):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    genes = set(adata.var.feature_name if "feature_name" in adata.var else adata.var_names)
    for ct, gs in markers.items():
        gs = [g for g in gs if g in genes]
        if gs:
            sc.tl.score_genes(adata, gs, score_name=f"score_{ct}", use_raw=False)
    score_cols = [c for c in adata.obs if c.startswith("score_")]
    best = adata.obs[score_cols].idxmax(axis=1).str.removeprefix("score_")
    adata.obs["marker_best"] = best
    return (adata.obs.assign(supported=adata.obs[label_col] == best)
                     .groupby(label_col).supported.mean())